# H6: Gemini: универсальная VLM против специализированной OCR

**Проверяемая гипотеза:** На общих 200 test-примерах уступает PP-OCRv6 small не более чем на 2 п.п.

Мультимодальная генеративная модель получает изображение и инструкцию, возвращает текст. Это другой класс задачи декодирования, чем CTC. Точные слои, число параметров и состав обучения Gemini 3.8 Flash публично не раскрыты: внутреннюю архитектуру не выдумываем.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL_V2.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [ ]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation_v2 import (manifest,run_local,run_gemini,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('READ ONLY: historical results, paid requests are disabled.')


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [ ]:
labels=manifest('shared_200' if 'gemini'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

## Сохранённый реальный прогон: новые запросы отключены

Исторический запуск уже состоялся и находится в research/executed_v2. Эта версия читает его настоящие ответы из CSV. Она не проверяет подключение, не повторяет ошибки и не отправляет изображения. Платная серия остановлена пользователем; сначала требуется выбор модели и бюджета.

In [ ]:
from captcha_lab.evaluation_v2 import load_predictions
frames={name:load_predictions('gemini',name) for name in ['shared_200','target']}
print('Saved real predictions loaded; no API calls.')

In [ ]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    destination=RESULTS/('gemini')/(dataset+'_scored.csv')
    # Read-only review: do not overwrite historical results.
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [ ]:
summary=all_summaries()
display(summary[summary.model.eq('gemini')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H6')])
display(paired[paired.hypothesis.eq('H6')] if not paired.empty else paired)